In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 데이터 불러오기 & 리샘플링

In [150]:
price_data_4h = pd.read_csv("./data/ETH_4h_data_all.csv", encoding="utf-8-sig", index_col=0, parse_dates=True)
price_data_4h.index.name = "Datetime"
price_data_4h.index = price_data_4h.index.tz_localize('Asia/Seoul').tz_convert('UTC').tz_localize(None)
price_data_4h.columns = [col.capitalize() for col in price_data_4h.columns]
price_data_4h = price_data_4h[["Open", "High", "Low", "Close", "Volume"]]
price_data_4h.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-09-26 08:00:00,324500.0,328500.0,323500.0,327000.0,0.0070
2017-09-26 12:00:00,327000.0,328000.0,322500.0,323000.0,0.0086
2017-09-26 16:00:00,323000.0,324000.0,321000.0,322000.0,0.0088
2017-09-26 20:00:00,321500.0,324500.0,320000.0,321500.0,0.0088
2017-09-27 00:00:00,321000.0,325500.0,320000.0,324500.0,0.0083


In [15]:
# 리샘플링 규칙 정의
ohlc_dict = {
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}

# 4시간봉 데이터를 일봉으로 리샘플링
price_data_1d = price_data_4h.resample('D').apply(ohlc_dict)
price_data_1d.dropna(inplace=True)
price_data_1d.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-09-26,324500.0,328500.0,320000.0,321500.0,0.0332
2017-09-27,321000.0,342500.0,320000.0,342500.0,0.0507
2017-09-28,342000.0,346500.0,327000.0,332500.0,0.0497
2017-09-29,331000.0,333500.0,312500.0,327500.0,0.0523
2017-09-30,327500.0,341500.0,327000.0,341000.0,0.0525


# 피쳐 추가

In [151]:
import pandas_ta as ta

# 4시간봉 데이터에 피처 추가
features_4h = price_data_4h.copy()
features_4h.ta.rsi(length=14, append=True, col_names=('RSI_14_4H',))
features_4h.ta.macd(fast=12, slow=26, signal=9, append=True, col_names=('MACD_12_26_9_4H', 'MACDh_12_26_9_4H', 'MACDs_12_26_9_4H'))
features_4h.ta.bbands(length=20, std=2, append=True, col_names=('BBL_20_2.0_4H', 'BBM_20_2.0_4H', 'BBU_20_2.0_4H', 'BBB_20_2.0_4H', 'BBP_20_2.0_4H'))

# 일봉 데이터에 피처 추가
features_1d = price_data_1d.copy()
features_1d.ta.rsi(length=14, append=True, col_names=('RSI_14_1D',))
features_1d.ta.sma(length=50, append=True, col_names=('SMA_50_1D',))
features_1d.ta.adx(length=14, append=True, col_names=('ADX_14_1D', 'DMP_14_1D', 'DMN_14_1D'))

# 불필요한 컬럼 및 NaN 값 제거
features_4h.drop(['Open', 'High', 'Low', 'Close', 'Volume'], axis=1, inplace=True)
features_1d.drop(['Open', 'High', 'Low', 'Close', 'Volume'], axis=1, inplace=True)

In [152]:
# 일봉 피처를 4시간봉 인덱스에 맞게 재정렬하고 ffill로 채우기
aligned_features_1d = features_1d.reindex(features_4h.index, method='ffill')

# 4시간봉 원본 데이터와 두 타임프레임의 피처를 결합
final_features = pd.concat([price_data_4h, features_4h, aligned_features_1d], axis=1)
final_features.dropna(inplace=True)

In [153]:
final_features.head()

,Open,High,Low,Close,Volume,RSI_14_4H,MACD_12_26_9_4H,MACDh_12_26_9_4H,MACDs_12_26_9_4H,BBL_20_2.0_4H,BBM_20_2.0_4H,BBU_20_2.0_4H,BBB_20_2.0_4H,BBP_20_2.0_4H,RSI_14_1D,SMA_50_1D,ADX_14_1D,DMP_14_1D,DMN_14_1D
Datetime,,,,,,,,,,,,,,,,,,,
2017-11-17 00:00:00,364500.0,369700.0,362400.0,368750.0,10850.958490,48.876348,1316.848858,-2157.013084,3473.861942,360951.222780,374240.0,387528.777220,7.101741,0.293435,55.070315,350460.0,16.201964,21.109434,15.604106
2017-11-17 04:00:00,368400.0,368800.0,362300.0,363650.0,9879.929128,43.742110,612.246354,-2289.292471,2901.538825,361455.970374,374350.0,387244.029626,6.888756,0.085079,55.070315,350460.0,16.201964,21.109434,15.604106
2017-11-17 08:00:00,363600.0,367500.0,362500.0,365100.0,6881.206523,45.495162,168.899704,-2186.111296,2355.011000,361711.993120,374412.5,387113.006880,6.784232,0.133381,55.070315,350460.0,16.201964,21.109434,15.604106
2017-11-17 12:00:00,365100.0,370000.0,365100.0,368050.0,10503.492734,48.978557,54.951192,-1840.047846,1894.999039,361032.131022,373965.0,386897.868978,6.916620,0.271319,55.070315,350460.0,16.201964,21.109434,15.604106
2017-11-17 16:00:00,368000.0,368000.0,365000.0,365150.0,7907.621750,45.874701,-266.289963,-1729.031201,1462.741239,360035.521828,373527.5,387019.478172,7.224088,0.189538,55.070315,350460.0,16.201964,21.109434,15.604106


# 피쳐 추가2

In [154]:
import pandas_ta as ta

# 4시간봉 데이터에 피처 추가
features_4h = price_data_4h.copy()
features_4h.columns = features_4h.columns

# 일봉 데이터에 피처 추가
features_1d = price_data_1d.copy()
features_1d.columns = features_1d.columns + '_1D'

aligned_features_1d = features_1d.reindex(features_4h.index, method='ffill')
final_features = pd.concat([features_4h, aligned_features_1d], axis=1)
final_features.dropna(inplace=True)

final_features

,Open,High,Low,Close,Volume,Open_1D,High_1D,Low_1D,Close_1D,Volume_1D
Datetime,,,,,,,,,,
2017-09-26 08:00:00,324500.0,328500.0,323500.0,327000.0,0.007000,324500.0,328500.0,320000.0,321500.0,0.033200
2017-09-26 12:00:00,327000.0,328000.0,322500.0,323000.0,0.008600,324500.0,328500.0,320000.0,321500.0,0.033200
2017-09-26 16:00:00,323000.0,324000.0,321000.0,322000.0,0.008800,324500.0,328500.0,320000.0,321500.0,0.033200
2017-09-26 20:00:00,321500.0,324500.0,320000.0,321500.0,0.008800,324500.0,328500.0,320000.0,321500.0,0.033200
2017-09-27 00:00:00,321000.0,325500.0,320000.0,324500.0,0.008300,321000.0,342500.0,320000.0,342500.0,0.050700
...,...,...,...,...,...,...,...,...,...,...
2025-08-16 20:00:00,6126000.0,6163000.0,6115000.0,6146000.0,3554.292114,6215000.0,6257000.0,6063000.0,6146000.0,51853.301374
2025-08-17 00:00:00,6146000.0,6198000.0,6107000.0,6184000.0,9645.929454,6146000.0,6342000.0,6107000.0,6302000.0,50363.536960
2025-08-17 04:00:00,6184000.0,6238000.0,6161000.0,6191000.0,11904.463331,6146000.0,6342000.0,6107000.0,6302000.0,50363.536960


# 데이터 전처리

In [155]:
from sklearn.preprocessing import MinMaxScaler
import joblib

# 시간 순서에 따른 데이터 분할
train_data = final_features.loc[:'2024-06']
validation_data = final_features.loc['2024-07':'2024-12']
test_data = final_features.loc['2025':]

# 스케일러 훈련 및 적용
scaler = MinMaxScaler()
scaled_train_features = scaler.fit_transform(train_data)
scaled_validation_features = scaler.transform(validation_data)
scaled_test_features = scaler.transform(test_data)

joblib.dump(scaler, './lstm/best_mtf_scaler_eth_ver1.pkl')

# 3D 시퀀스 데이터 생성
def create_sequences(data, lookback_window):
    X, y = [], []
    for i in range(lookback_window, len(data)):
        X.append(data[i-lookback_window:i, :])
        y.append(data[i, 3]) # 종가(인덱스 3)를 임시 타겟으로 설정
    return np.array(X), np.array(y)

lookback = 30 # 30
X_train, _ = create_sequences(scaled_train_features, lookback)
X_val, _ = create_sequences(scaled_validation_features, lookback)
X_test, _ = create_sequences(scaled_test_features, lookback)

# 라벨(정답지)

In [156]:
def get_triple_barrier_labels(prices, entries, profit_take_pct, stop_loss_pct, max_hold_periods):
    """
    금융 시계열 데이터 레이블링을 위한 삼중 장벽 기법을 구현합니다.

    Args:
        prices (pd.Series): 가격 시계열 (예: 'Close').
        entries (pd.DatetimeIndex): 거래를 시작하는 타임스탬프.
        profit_take_pct (float): 익절 장벽의 비율.
        stop_loss_pct (float): 손절 장벽의 비율.
        max_hold_periods (int): 최대 포지션 보유 기간.

    Returns:
        pd.Series: 각 진입 시점에 대한 레이블 (1: 익절, -1: 손절, 0: 기간 만료).
    """
    results = pd.Series(index=entries, dtype='int8')
    
    for entry_time in entries:
        entry_price = prices.loc[entry_time]
        
        # 1. 수직 장벽 설정
        end_of_window_idx = prices.index.get_loc(entry_time) + max_hold_periods
        if end_of_window_idx >= len(prices.index):
            end_of_window_idx = len(prices.index) - 1
        end_of_window = prices.index[end_of_window_idx]
        
        # 2. 수평 장벽 설정
        profit_take_level = entry_price * (1 + profit_take_pct)
        stop_loss_level = entry_price * (1 - stop_loss_pct)
        
        # 3. 장벽 도달 시간 계산
        price_path = prices.loc[entry_time:end_of_window]
        
        profit_hit_time = price_path[price_path >= profit_take_level].first_valid_index()
        stop_loss_hit_time = price_path[price_path <= stop_loss_level].first_valid_index()
        
        # 4. 레이블 결정
        if profit_hit_time is not None and (stop_loss_hit_time is None or profit_hit_time <= stop_loss_hit_time):
            results.loc[entry_time] = 1
        elif stop_loss_hit_time is not None:
            results.loc[entry_time] = -1
        else:
            results.loc[entry_time] = 0
            
    return results

# 레이블링을 위한 진입 시점 정의 (시퀀싱으로 인해 앞부분 제외)
# 여기서는 모든 시점을 진입 후보로 간주하여 레이블링합니다.
train_entries = train_data.index[lookback:]
val_entries = validation_data.index[lookback:]
test_entries = test_data.index[lookback:]

# 각 데이터셋에 대한 레이블 생성
labels_train = get_triple_barrier_labels(
    prices=train_data['Close'], entries=train_entries,
    profit_take_pct=0.02, stop_loss_pct=0.01, max_hold_periods=12 # 12 * 4H = 2일
)
labels_val = get_triple_barrier_labels(
    prices=validation_data['Close'], entries=val_entries,
    profit_take_pct=0.02, stop_loss_pct=0.01, max_hold_periods=12
)
labels_test = get_triple_barrier_labels(
    prices=test_data['Close'], entries=test_entries,
    profit_take_pct=0.02, stop_loss_pct=0.01, max_hold_periods=12
)

# 모델

In [157]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Bidirectional, Conv1D, MaxPooling1D
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight

# 삼중 장벽 기법으로 생성된 레이블 (labels_train, labels_val, labels_test)
# 레이블을 원-핫 인코딩으로 변환: -1 -> , 0 -> , 1 -> 
y_train_cat = to_categorical(labels_train + 1, num_classes=3)
y_val_cat = to_categorical(labels_val + 1, num_classes=3)
y_test_cat = to_categorical(labels_test + 1, num_classes=3)

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_train), # 고유 클래스: [-1, 0, 1]
    y=labels_train
)

class_weights_dict = dict(enumerate(weights))

model = Sequential([
    LSTM(units=64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(units=32, activation='relu'),
    Dense(units=3, activation='softmax')
])


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

d:\desktop\wafflestudio\hasha\coin\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 64)             │        19,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,379 (83.51 KB)

 Trainable params: 21,379 (83.51 KB)

 Non-trainable params: 0 (0.00 B)

# 훈련

In [158]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_cb = ModelCheckpoint("./lstm/best_mtf_model_eth_ver1.h5", save_best_only=True)
early_stopping_cb = EarlyStopping(patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    callbacks=[checkpoint_cb, early_stopping_cb],
    # class_weight=class_weights_dict
)

Epoch 1/100
227/231 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5419 - loss: 0.9481

231/231 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5453 - loss: 0.9245 - val_accuracy: 0.5428 - val_loss: 0.9231
Epoch 2/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5466 - loss: 0.9001 - val_accuracy: 0.5428 - val_loss: 0.9481
Epoch 3/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5467 - loss: 0.8875 - val_accuracy: 0.5428 - val_loss: 0.9288
Epoch 4/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5462 - loss: 0.8826 - val_accuracy: 0.5428 - val_loss: 0.9339
Epoch 5/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5471 - loss: 0.8791 - val_accuracy: 0.5428 - val_loss: 0.9444
Epoch 6/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5471 - loss: 0.8763 - val_accuracy: 0.5428 - val_loss: 0.9609
Epoch 7/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5465 - loss: 0.8762 - val_accuracy: 0.5428 - val_loss: 0.9388
Epoch 8/100
231/231 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5469 - loss: 0.8745 - val_accuracy: 0.542

# 테스트

# 검증 셋으로 threshold 값 찾기

In [159]:
from sklearn.metrics import f1_score

from keras.models import load_model

# ---------------------------------
# 1. 저장된 모델 불러오기
# ---------------------------------
# 'my_lstm_model.h5' 부분에 실제 저장한 모델 파일 경로를 입력하세요.
print("모델을 불러오는 중입니다...")
model = load_model('./lstm/best_mtf_model_eth_ver1.h5')
print("모델 로딩 완료!")

# 1. 훈련된 모델로 검증셋(X_val)에 대한 예측 확률 계산
print("검증셋에 대한 예측을 수행합니다...")
val_predictions = model.predict(X_val)

# '익절'(P=2) 클래스에 대한 예측 확률과 실제 정답을 추출
# y_val_cat은 to_categorical로 변환된 검증셋의 정답 레이블입니다.
prob_profit_val = val_predictions[:, 2] 
y_true_val = y_val_cat[:, 2]

# 2. 최적의 Threshold를 찾기 위한 반복문
best_f1 = -1.0
best_threshold = 0

# 0.05부터 0.95까지 0.01 간격으로 모든 Threshold 후보를 테스트
for threshold in np.arange(0.05, 0.95, 0.01):
    
    # 현재 Threshold를 기준으로 예측값을 0 또는 1로 변환
    y_pred = (prob_profit_val > threshold).astype(int)
    
    # F1-Score 계산
    current_f1 = f1_score(y_true_val, y_pred)
    
    # 만약 현재 F1-Score가 역대 최고 점수라면?
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print("\n--- 최적 Threshold 탐색 완료 ---")
print(f"최적 Entry Threshold: {best_threshold:.2f}")
print(f"해당 Threshold에서의 F1-Score: {best_f1:.4f}")

# '손절'(P=0) 클래스에 대한 예측 확률과 실제 정답을 추출
# y_val_cat은 to_categorical로 변환된 검증셋의 정답 레이블입니다.
prob_loss_val = val_predictions[:, 0] 
y_true_val_loss = y_val_cat[:, 0]

# 2. 최적의 Threshold를 찾기 위한 반복문
best_f1_exit = -1.0
best_threshold_exit = 0

# 0.05부터 0.95까지 0.01 간격으로 모든 Threshold 후보를 테스트
for threshold in np.arange(0.05, 0.95, 0.01):
    
    # 현재 Threshold를 기준으로 예측값을 0 또는 1로 변환
    y_pred_loss = (prob_loss_val > threshold).astype(int)
    
    # F1-Score 계산
    current_f1_exit = f1_score(y_true_val_loss, y_pred_loss)
    
    # 만약 현재 F1-Score가 역대 최고 점수라면?
    if current_f1_exit > best_f1_exit:
        best_f1_exit = current_f1_exit
        best_threshold_exit = threshold

print("\n--- 최적 Threshold 탐색 완료 (손절/Exit) ---")
print(f"최적 Exit Threshold: {best_threshold_exit:.2f}")
print(f"해당 Threshold에서의 F1-Score: {best_f1_exit:.4f}")

모델을 불러오는 중입니다...


모델 로딩 완료!
검증셋에 대한 예측을 수행합니다...
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

--- 최적 Threshold 탐색 완료 ---
최적 Entry Threshold: 0.33
해당 Threshold에서의 F1-Score: 0.5471

--- 최적 Threshold 탐색 완료 (손절/Exit) ---
최적 Exit Threshold: 0.05
해당 Threshold에서의 F1-Score: 0.7037


## 테스트셋으로 확인

In [205]:
# 테스트 데이터에 대한 예측 수행
predictions = model.predict(X_test)

# 확률을 신호로 변환 (예시 로직)
prob_profit = predictions[:, 2] # P(익절)
prob_loss = predictions[:, 0]   # P(손절)

# '익절' 확률의 최댓값과 평균값 확인
print(f"최대 익절 예측 확률: {np.max(prob_profit):.4f}")
print(f"평균 익절 예측 확률: {np.mean(prob_profit):.4f}") 


print(f"최대 손절 예측 확률: {np.max(prob_loss):.4f}")
print(f"평균 손절 예측 확률: {np.mean(prob_loss):.4f}")

# entry_threshold = 0.16
# exit_threshold = 0.24
entry_threshold = 0.342
exit_threshold = 0.562

# test_data의 길이에 맞는 불리언 배열 생성
entries = pd.Series(False, index=test_data.index)
exits = pd.Series(False, index=test_data.index)

# 예측 결과가 시작되는 인덱스부터 신호 생성
signal_index = test_data.index[lookback:]

entries.loc[signal_index] = prob_profit > entry_threshold
exits.loc[signal_index] = prob_loss > exit_threshold


# 테스트 기간의 종가 데이터 사용
price_close_test = test_data['Close'][lookback:]
entries = entries[lookback:]
exits = exits[lookback:]

# 진입 우선
conflicting_signals = entries & exits
exits[conflicting_signals] = False

print(len(entries), entries.sum(), exits.sum())



42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
최대 익절 예측 확률: 0.3700
평균 익절 예측 확률: 0.3425
최대 손절 예측 확률: 0.5754
평균 손절 예측 확률: 0.5628
1342 672 553


In [206]:
import vectorbt as vbt

portfolio = vbt.Portfolio.from_signals(
    close=price_close_test,
    entries=entries,
    exits=exits,
    init_cash=1000000,
    fees=0.0005,      # 0.05% 수수료(maker)
    slippage=0.0005, # 0.05% 슬리피지
    freq='4h',        # 데이터 빈도 명시
    # stop_exit=True,   # 진입 시 청산 고려
    # size=100000,                # 거래당 100,000 KRW씩 진입 (최소 주문 금액 5,000원 이상)
    # size_type='value',           # size가 '금액' 기준임을 명시
    # accumulate=True
)
# 성과 통계 출력
print(portfolio.stats())

# 벤치마크(단순 보유)와 함께 성과 시각화
portfolio.plot().show()

Start                         2025-01-06 00:00:00
End                           2025-08-17 12:00:00
Period                          223 days 16:00:00
Start Value                             1000000.0
End Value                          1197316.281953
Total Return [%]                        19.731628
Benchmark Return [%]                    16.768575
Max Gross Exposure [%]                      100.0
Total Fees Paid                       5172.140017
Max Drawdown [%]                         43.86011
Max Drawdown Duration            98 days 16:00:00
Total Trades                                    5
Total Closed Trades                             5
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                                 60.0
Best Trade [%]                          11.682846
Worst Trade [%]                         -3.778185
Avg Winning Trade [%]                    7.651024
Avg Losing Trade [%]                    -1.946947


# 테스트 - 전체 기간

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from keras.models import load_model
import joblib
import pandas_ta as ta

test_data = "./data/BTC_4h_data_all.csv"
test_data = "./data/recent_candles_test.csv"

price_data_4h = pd.read_csv(test_data, encoding="utf-8-sig", index_col=0, parse_dates=True)
# 리샘플링 규칙 정의
ohlc_dict = {
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}

# 4시간봉 데이터를 일봉으로 리샘플링
price_data_1d = price_data_4h.resample('D').apply(ohlc_dict)

# ---------------------------------
# 1. 저장된 모델 불러오기
# ---------------------------------
# 'my_lstm_model.h5' 부분에 실제 저장한 모델 파일 경로를 입력하세요.
print("모델을 불러오는 중입니다...")
model = load_model('./lstm/best_mtf_model_eth_ver1.h5')
print("모델 로딩 완료!")

entry_threshold = 0.22
exit_threshold = 0.55

# ==============================================================================
# 2. 전체 기간 데이터 준비 (KeyError 해결 최종안)
# ==============================================================================
# (1) 피처를 처음부터 다시 생성하여 데이터 일관성을 보장합니다.

# 4시간봉 피처 생성: 원본 가격 데이터(OHLCV 포함)를 복사해서 시작합니다.
print("4시간봉 피처를 생성합니다...")
features_4h_new = price_data_4h.copy()
features_4h_new.ta.rsi(length=14, append=True, col_names=('RSI_14_4H',))
features_4h_new.ta.macd(fast=12, slow=26, signal=9, append=True, col_names=('MACD_12_26_9_4H', 'MACDh_12_26_9_4H', 'MACDs_12_26_9_4H'))
features_4h_new.ta.bbands(length=20, std=2, append=True, col_names=('BBL_20_2.0_4H', 'BBM_20_2.0_4H', 'BBU_20_2.0_4H', 'BBB_20_2.0_4H', 'BBP_20_2.0_4H'))

# 일봉 피처 생성
print("일봉 피처를 생성합니다...")
features_1d_new = price_data_1d.copy()
features_1d_new.ta.rsi(length=14, append=True, col_names=('RSI_14_1D',))
features_1d_new.ta.sma(length=50, append=True, col_names=('SMA_50_1D',))
features_1d_new.ta.adx(length=14, append=True, col_names=('ADX_14_1D', 'DMP_14_1D', 'DMN_14_1D'))

# 사용할 일봉 지표 컬럼만 선택합니다 (OHLCV 중복 방지)
daily_indicator_cols = ['RSI_14_1D', 'SMA_50_1D', 'ADX_14_1D', 'DMP_14_1D', 'DMN_14_1D']
features_1d_to_merge = features_1d_new[daily_indicator_cols]


# (2) 두 시간대 피처 결합
# 4시간봉 피처(OHLCV 포함)를 기준으로 일봉 지표를 합칩니다.
print("다중 시간대 피처를 결합합니다...")
final_features = pd.merge(features_4h_new, features_1d_to_merge, left_index=True, right_index=True, how='left')
final_features.fillna(method='ffill', inplace=True)
final_features.dropna(inplace=True)
print("데이터 준비 완료!")


# (3) Scaler 적용 (이전과 동일한 정렬 로직 사용)
scaler = joblib.load('./lstm/best_mtf_scaler_ver1.1.pkl')

fit_feature_names = scaler.feature_names_in_
final_features_aligned = final_features[fit_feature_names]
scaled_features_full = scaler.transform(final_features_aligned)


# (4) 시퀀스 데이터 생성 (lookback 값 확인!)
lookback = 30 # 훈련 시 사용했던 값으로 반드시 통일해야 합니다!
X_full = []
for i in range(lookback, len(scaled_features_full)):
    X_full.append(scaled_features_full[i-lookback:i])
X_full = np.array(X_full)


# ==============================================================================
# 3. 전체 기간에 대한 예측 수행 (★★★ 이 부분이 추가되었습니다 ★★★)
# ==============================================================================
print(f"총 {len(X_full)}개의 시퀀스에 대한 예측을 수행합니다...")
predictions_full = model.predict(X_full)

# 예측 결과를 각 확률 변수에 할당
prob_loss_full = predictions_full[:, 0]
prob_hold_full = predictions_full[:, 1]
prob_profit_full = predictions_full[:, 2]
print("예측 완료!")

print(prob_profit_full.mean(), prob_profit_full.max())
print(prob_hold_full.mean(), prob_hold_full.max())
print(prob_loss_full.mean(), prob_loss_full.max())

# ==============================================================================
# 4. 시그널 생성 및 VectorBT 백테스팅
# ==============================================================================
# 예측이 시작되는 시점에 맞춰 인덱스를 정렬합니다.
signal_index_full = final_features.index[lookback:]

# 시그널 생성을 위한 빈 Series 생성
entries_full = pd.Series(False, index=price_data_4h.index) # 기준이 되는 4시간봉 인덱스 사용
exits_full = pd.Series(False, index=price_data_4h.index)


# loc를 사용하여 정확한 위치에 시그널 할당
entries_full.loc[signal_index_full] = prob_profit_full > entry_threshold
exits_full.loc[signal_index_full] = prob_loss_full > exit_threshold

# # 진입과 청산 신호가 겹칠 경우 진입을 우선 (ipynb 로직 반영)
# conflicting_signals_full = entries_full & exits_full
# exits_full[conflicting_signals_full] = False


# 백테스팅 실행 (ipynb의 설정값 반영)
full_portfolio = vbt.Portfolio.from_signals(
    close=price_data_4h['Close'], # 백테스팅 기준 가격: 4시간봉 종가
    entries=entries_full,
    exits=exits_full,
    init_cash=1000000,
    fees=0.0005,      # 0.5% 수수료
    slippage=0.0005, # 0.05% 슬리피지
    freq='4h',        # 데이터 빈도 명시
    # stop_exit=True,   # 진입 시 청산 고려
    # size=0.5,                # 거래당 100,000 KRW씩 진입 (최소 주문 금액 5,000원 이상)
    # size_type='percent'           # size가 '금액' 기준임을 명시
)

# 최종 성과 출력
print("\n--- 전체 기간 최종 백테스팅 결과 ---")
print(full_portfolio.stats())

# 누적 수익률 그래프 시각화
full_portfolio.plot().show()

모델을 불러오는 중입니다...
모델 로딩 완료!
4시간봉 피처를 생성합니다...
일봉 피처를 생성합니다...
다중 시간대 피처를 결합합니다...
데이터 준비 완료!
총 230개의 시퀀스에 대한 예측을 수행합니다...
1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step

C:\Users\endermaru\AppData\Local\Temp\ipykernel_34840\4088253652.py:63: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
예측 완료!
0.22349018 0.48631933
0.21862337 0.35821834
0.5578865 0.8809723

--- 전체 기간 최종 백테스팅 결과 ---
Start                         2025-05-26 08:00:00+00:00
End                           2025-08-26 04:00:00+00:00
Period                                 92 days 00:00:00
Start Value                                   1000000.0
End Value                                1103821.791356
Total Return [%]                              10.382179
Benchmark Return [%]                           2.077792
Max Gross Exposure [%]                            100.0
Total Fees Paid                            15724.042457
Max Drawdown [%]                               1.758394
Max Drawdown Duration                   8 days 08:00:00
Total Trades                                         15
Total Closed Trades                                  15
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]             

In [19]:
signals_df = pd.DataFrame(
    {
        'Profit_Prob': prob_profit_full,
        'Loss_Prob': prob_loss_full,
        'Entry_Signal': prob_profit_full > entry_threshold,
        'Exit_Signal': prob_loss_full > exit_threshold
    },
    index=signal_index_full # 예측 결과에 맞는 인덱스 사용
)
signals_df.to_csv("./data/prediction_signals.csv", encoding='utf-8-sig', index=True)

# 이상적인 전략

In [21]:
import pandas as pd
import vectorbt as vbt
import numpy as np

# --- 사전 준비 (이전 단계에서 이미 실행되었다고 가정) ---
# test_data: 훈련 데이터프레임 (OHLCV 및 피처 포함)
# labels_train: 삼중 장벽 기법으로 생성된 훈련 데이터의 레이블 (-1, 0, 1)
# lookback: 시퀀스 생성을 위한 lookback 기간 (예: 50)
# ---------------------------------------------------------

# 1. 백테스팅에 사용할 가격 데이터 준비
# labels_train은 lookback 기간 이후부터 생성되었으므로, 가격 데이터도 동일하게 맞춰줍니다.
price_close_test = test_data['Close'][lookback:]

# labels_test의 인덱스와 가격 데이터의 인덱스가 일치하는지 확인
# (get_triple_barrier_labels 함수에서 올바르게 생성했다면 일치해야 합니다)
aligned_labels_test = labels_test.reindex(price_close_test.index).dropna()
price_close_test = price_close_test.reindex(aligned_labels_test.index)

# 2. '정답' 레이블을 'entries'와 'exits' 신호로 변환
# '익절'(1)이 발생한 시점을 진입 신호로 간주
entries_from_labels = (aligned_labels_test == 1)

# '손절'(-1)이 발생한 시점을 청산 신호로 간주
exits_from_labels = (aligned_labels_test == -1)

# 3. vectorbt 포트폴리오 실행
# "Perfect Foresight Strategy" (완벽한 예측 전략)
perfect_portfolio = vbt.Portfolio.from_signals(
    close=price_close_test,
    entries=entries_from_labels,
    exits=exits_from_labels,
    init_cash=100000,
    fees=0.001,
    slippage=0.0005,
    freq='4h'
)

# 4. 결과 확인
print("--- 훈련 데이터와 정답 레이블을 이용한 백테스팅 결과 (이론적 상한선) ---")
print(perfect_portfolio.stats())

# 시각화
perfect_portfolio.plot(title="Performance with Perfect Foresight (testing Data)").show()

TypeError: string indices must be integers, not 'str'